In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

print("=" * 80)
print(" GENERATING E-COMMERCE SAMPLE DATA WITH INTENTIONAL QUALITY ISSUES")
print("=" * 80)

# ========== STEP 1: GENERATE CUSTOMERS.CSV ==========
print("\n📊 STEP 1: Generating customers.csv (10,000 rows)...")

num_customers = 10000
customer_ids = list(range(1, num_customers + 1))

# Realistic names
first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 
               'William', 'Elizabeth', 'David', 'Barbara', 'Richard', 'Susan', 'Joseph', 'Jessica',
               'Thomas', 'Sarah', 'Charles', 'Karen', 'Daniel', 'Nancy', 'Matthew', 'Lisa']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Hernandez', 'Lopez', 'Gonzalez', 'Wilson', 'Anderson',
              'Thomas', 'Taylor', 'Moore', 'Jackson', 'Martin', 'Lee', 'Perez', 'Thompson', 'White']

customer_names = [f"{random.choice(first_names)} {random.choice(last_names)}" for _ in range(num_customers)]
emails = [f"{name.lower().replace(' ', '.')}_{i}@example.com" for i, name in enumerate(customer_names)]

countries = ['USA', 'UK', 'Canada', 'Germany', 'France', 'Australia', 'India', 'China', 'Japan', 'Brazil']
customer_countries = [random.choice(countries) for _ in range(num_customers)]

# Signup dates (2020-2024)
start_date = datetime(2020, 1, 1)
end_date = datetime(2024, 12, 31)
date_range = (end_date - start_date).days
signup_dates = [start_date + timedelta(days=random.randint(0, date_range)) for _ in range(num_customers)]

# Customer segments
segments = ['Premium', 'Standard', 'Basic']
segment_weights = [0.15, 0.55, 0.30]
customer_segments = random.choices(segments, weights=segment_weights, k=num_customers)

# Lifetime value correlated with segment
lifetime_values = []
for seg in customer_segments:
    if seg == 'Premium':
        ltv = round(random.uniform(5000, 50000), 2)
    elif seg == 'Standard':
        ltv = round(random.uniform(1000, 5000), 2)
    else:
        ltv = round(random.uniform(100, 1000), 2)
    lifetime_values.append(ltv)

customers_df = pd.DataFrame({
    'customer_id': customer_ids,
    'customer_name': customer_names,
    'email': emails,
    'country': customer_countries,
    'signup_date': [d.strftime('%Y-%m-%d') for d in signup_dates],
    'customer_segment': customer_segments,
    'lifetime_value': lifetime_values
})

# INJECT QUALITY ISSUES
print("\n🔴 Injecting customer quality issues:")

# Issue 1: 50 NULL emails
null_email_indices = random.sample(range(len(customers_df)), 50)
customers_df.loc[null_email_indices, 'email'] = None
print(f"  ✓ Issue 1: Added 50 NULL emails (completeness check)")

# Issue 2: 10 duplicate customer_ids
duplicate_indices = random.sample(range(100, 500), 10)
duplicate_rows = customers_df[customers_df['customer_id'].isin(duplicate_indices)].copy()
duplicate_rows['customer_name'] = duplicate_rows['customer_name'] + ' (Duplicate)'
customers_df = pd.concat([customers_df, duplicate_rows], ignore_index=True)
print(f"  ✓ Issue 2: Added 10 duplicate customer_id rows (uniqueness check)")

customers_df = customers_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save to Volume
customer_path = '/Volumes/workspace/dev/medallion_data/customers.csv'
customers_df.to_csv(customer_path, index=False)

print(f"\n✅ Customers.csv generated:")
print(f"   Total rows: {len(customers_df):,}")
print(f"   NULL emails: {customers_df['email'].isna().sum()}")
print(f"   Duplicate customer_ids: {customers_df['customer_id'].duplicated().sum()}")
print(f"   Saved to: {customer_path}")

# Store valid customer IDs for orders
valid_customer_ids = customers_df['customer_id'].drop_duplicates().tolist()

print(f"\nCustomers generation complete! ✅")